In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from lib.layers import Residual, UnpackGrid, MultiBatchConv2d
from lib.quantumsearch import FitnessFunction, OneToManyNetwork, SoftHardSearch, SoftSearch
from lib.quantumsearch import TransitionFunction

In [ ]:
dataset = load_dataset("HuggingFaceM4/RAVEN", "center_single")


In [ ]:
columns_to_remove = ['structure', 'meta_matrix', 'meta_target', 'meta_structure', 'metadata']
search_dataset = dataset.remove_columns(columns_to_remove)
print(search_dataset)


In [ ]:
class RAVENDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        self.transform = transforms.Compose([transforms.ToTensor(),])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        panels = sample['panels']
        choices = sample['choices']
        target = sample['target']
        images = panels + choices
        images = [self.transform(img) for img in images]
        stacked = torch.cat(images, dim=0)
        return stacked, target

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
train_dataset = RAVENDataset(search_dataset['train'])
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, generator= torch.Generator(device=device))

val_dataset = RAVENDataset(search_dataset['validation'])
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = True, generator= torch.Generator(device=device))

In [ ]:
train_dataset.__getitem__(0)


In [ ]:

class ResNetBlock(nn.Module):
    """Basic redisual block."""

    def __init__(
        self,
        num_input_filters: int,
        num_output_filters: int

    ) -> None:
        super().__init__()

        self.conv_block1 = nn.Sequential(
            MultiBatchConv2d(
                in_channels = num_input_filters,
                out_channels = num_input_filters,
                kernel_size = 3,
                stride = 1,
                padding = 1,
                bias = False,
            ),
            # nn.BatchNorm2d(num_features=num_filters),

            nn.ReLU(),
        )

        self.conv_block2 = nn.Sequential(
            MultiBatchConv2d(
                in_channels = num_input_filters,
                out_channels = num_output_filters,
                kernel_size = 3,
                stride = 1,
                padding = 1,
                bias = False,
            ),
            # nn.BatchNorm2d(num_features=num_filters),
        )
        self.conv_block3 = MultiBatchConv2d(
                in_channels = num_input_filters,
                out_channels = num_output_filters,
                kernel_size = 1,
                stride = 1,
                bias = False,
            )
        self.layer_norm1 = None
        self.layer_norm2 = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.conv_block3(x)
        out = self.conv_block1(x)

        _,_,C,H,W = out.shape
        if self.layer_norm1 is None:
            self.layer_norm1 = nn.LayerNorm([C, H, W])
        out = self.layer_norm1(out)
        out = self.conv_block2(out)
        _,_,C,H,W = out.shape
        if self.layer_norm2 is None:
            self.layer_norm2 = nn.LayerNorm([C, H, W])
        out = self.layer_norm2(out)
        out += residual
        out = F.relu(out)
        return out


In [ ]:

num_filters = 16

encoder = nn.Sequential(
    MultiBatchConv2d(16, num_filters, 3, 1),
    nn.ReLU(),
)

search = SoftSearch(
    transition = TransitionFunction(OneToManyNetwork(
            nn.Sequential(
                ResNetBlock(num_input_filters = num_filters, num_output_filters = 3*num_filters),
                UnpackGrid(3) # Batch, ...,  3 * H -> Batch, ..., H, 3
            )
        ),
    ),
    fitness=FitnessFunction(
        OneToManyNetwork(
            nn.Sequential(

               ResNetBlock(num_input_filters = num_filters, num_output_filters = 3),
               UnpackGrid(3) # Batch, ...,  3 * H -> Batch, ..., 1, 3
            )
        ),
    ),
    max_depth=5,
    beam_width=3,
    branching_width=3,

)

decoder = nn.Sequential(
            nn.AdaptiveAvgPool2d((16, 16)),
            nn.Conv2d(
                in_channels=32,
                out_channels=1,
                kernel_size=1,
                bias=False,
            ),
            # nn.BatchNorm2d(num_features=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 8),
        )
model = nn.Sequential(encoder,
    search,
    decoder)
model.to(device)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f" Total number of parameters: {total_params}")

In [ ]:
def validate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    avg_loss = val_loss / len(val_loader)
    acc = 100. * correct / total
    return avg_loss, acc

In [ ]:
learning_rate = 1e-4
lambda_l2 = 1e-4
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr =learning_rate, betas=(0.9, 0.999), eps=1e-08)
with device:
    for epoch in range(20):
        model.train()
        for batch, targets in train_loader:
            batch, targets = batch.to(device), targets.to(device)
            y_pred = model(batch)
            loss = criterion(y_pred, targets)
            _, prediction = torch.max(y_pred, 1)
            accuracy = (targets == prediction).sum().float()/len(targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        print(f"Epoch {epoch+1},  loss: {loss.item(): .6f}. accuracy: {accuracy: .2f}%, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.2f}%")



In [ ]:

num_filters = 32

encoder1 = nn.Sequential(
    MultiBatchConv2d(16, 32, 3, 1),
    nn.ReLU(),
)

search1 = SoftHardSearch(
    transition = TransitionFunction(OneToManyNetwork(
            nn.Sequential(
                ResNetBlock(num_input_filters = num_filters, num_output_filters = 3*num_filters),
                UnpackGrid(3) # Batch, ...,  3 * H -> Batch, ..., H, 3
            )
        ),
    ),
    fitness=nn.Sequential(
        ResNetBlock(num_input_filters = num_filters, num_output_filters = 1),
        # UnpackGrid(3) # Batch, ...,  3 * H -> Batch, ..., 1, 3
    ),
    max_depth=1,
    beam_width=3,
    branching_width=3,
    token_dims=2,

)

decoder1 = nn.Sequential(
            nn.AdaptiveAvgPool2d((16, 16)),
            nn.Conv2d(
                in_channels=32,
                out_channels=1,
                kernel_size=1,
                bias=False,
            ),
            # nn.BatchNorm2d(num_features=1),
            # nn.ReLU(),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 8),
        )
model1 = nn.Sequential(encoder1,
    search1,
    decoder1)
model1.to(device)
total_params = sum(p.numel() for p in model1.parameters())
print(f" Total number of parameters: {total_params}")

In [ ]:
def validate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    avg_loss = val_loss / len(val_loader)
    acc = 100. * correct / total
    return avg_loss, acc

In [ ]:
learning_rate = 1e-4
lambda_l2 = 1e-4
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model1.parameters(), lr =learning_rate, betas=(0.9, 0.999), eps=1e-08)
with device:
    for epoch in range(20):
        model1.train()
        for batch, targets in train_loader:
            batch, targets = batch.to(device), targets.to(device)
            y_pred = model1(batch)
            loss = criterion(y_pred, targets)
            _, prediction = torch.max(y_pred, 1)
            accuracy = (targets == prediction).sum().float()/len(targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        val_loss, val_acc = validate(model1, val_loader, criterion, device)
        print(f"Epoch {epoch+1},  loss: {loss.item(): .6f}. accuracy: {accuracy: .2f}%, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.2f}%")

